# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Schema URL:** [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- **Dataset DOI:** 10.71728/senscience.vq4a-28xa
- **License:** [Open Data Commons Attribution License (ODC-By) v1.0](https://opendatacommons.org/licenses/by/1-0/)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

First, list all available record sets in the dataset with their `@id`. Then print sample records using these IDs.

In [ ]:
# List all available record sets and their @id
record_sets = dataset.metadata.record_sets
print("Available Record Sets (@id):")
for rs in record_sets:
    print(f"  @id: {rs.id} | name: {rs.name}")

# For demonstration, pick the first record set
first_record_set_id = record_sets[0].id if record_sets else None

# Review fields for the first record set
if first_record_set_id:
    print(f"\nFields in Record Set {first_record_set_id}:")
    first_rs = [rs for rs in record_sets if rs.id == first_record_set_id][0]
    for field in first_rs.fields:
        print(f"  Field @id: {field.id} | name: {field.name} | type: {field.data_type}")

# Show some records from the first record set
if first_record_set_id:
    print(f"\nSample records from Record Set {first_record_set_id}:")
    for i, x in enumerate(dataset.records(record_set=first_record_set_id)):
        print(x)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we load all record sets, referencing each by its `@id`, and display column names for one record set.

In [ ]:
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns from the first available record set
if record_set_ids:
    print(f"Columns in {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    dataframes[record_set_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, select a numeric field by its `@id`, filter values above a threshold, normalize that field, and group records by a key field if available.

In [ ]:
# Choose a record set and field @id dynamically
rs_to_analyze = record_set_ids[0] if record_set_ids else None
df = dataframes.get(rs_to_analyze, pd.DataFrame())

# Find a numeric field in this record set
numeric_field_id = None
group_field_id = None
if rs_to_analyze:
    rs_obj = [rs for rs in dataset.metadata.record_sets if rs.id == rs_to_analyze][0]
    for f in rs_obj.fields:
        if f.data_type in ['Integer', 'Float', 'Number']:
            numeric_field_id = f.id
            break
    # Pick first non-numeric field as group field
    for f in rs_obj.fields:
        if f.data_type not in ['Integer', 'Float', 'Number']:
            group_field_id = f.id
            break

if numeric_field_id and not df.empty and numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found or DataFrame is empty.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below we plot the distribution of the numeric field for the selected record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not df.empty and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=30)
    plt.title(f"Distribution of {numeric_field_id} in {rs_to_analyze}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("Unable to plot: missing numeric field or data.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Dataset contains rich protein abundance and modification data from mass spectrometry experiments on extracellular vesicles.
- Data can be explored via record sets and fields referenced by their `@id`, ensuring reproducible access.
- Numeric fields allow normalization and grouping; the dataset supports filtering for further biological or statistical analyses.
- Visualizations provide insight into distributions and relationships within protein data.

Continue analysis with domain-specific questions, statistical modeling, or machine learning tasks as required.